# CALVIN: A Benchmark for Language-Conditioned Long-Horizon Robot Manipulation
### A Self-Learning Notebook (+ Research Improvements)

Based on: *Mees, Hermann, Rosete-Beas & Burgard, 2022 — arXiv:2112.03227 (IEEE RA-L)*

One rule throughout:

**Concept first, code second.**

CALVIN is a *benchmark*, not an algorithm. The real benchmark runs on PyBullet and can't run in Colab, so this notebook teaches the **concepts** with lightweight synthetic stand-ins. The final sections then propose and implement three research improvements you can discuss with your advisor.

What you will understand:

- why language-conditioned, long-horizon manipulation is hard
- CALVIN's multimodal observation and action spaces
- how natural language is encoded and grounded to actions
- learning from unstructured "play" data via goal relabeling
- the three difficulty tiers and the long-horizon chaining metric
- the multi-context imitation learning (MCIL) baseline and why it struggles
- **three research improvements with working code**

All code uses synthetic toy data so the notebook runs anywhere.

## 0. Learning Goals

By the end you should be able to:

1. Explain why goal-image / one-hot task specification doesn't scale, and why language does.
2. Represent CALVIN's multimodal observation and multi-option action spaces.
3. Encode language instructions into embeddings and condition a policy on them.
4. Implement goal relabeling on unstructured play data.
5. Build a language-conditioned policy and compute the long-horizon chaining metric.
6. Reproduce the difficulty-tier evaluation protocol (D->D, ABCD->D, ABC->D).
7. Implement three improvements: vision-language contrastive grounding, a hierarchical sub-goal planner, and progress-based dense rewards.

In [ ]:
# == Cell 0: Setup ==
# The real CALVIN needs PyBullet + the 24h dataset and can't run in Colab.
# This notebook teaches the CONCEPTS with lightweight synthetic stand-ins.

!pip install torch --quiet

import math, random
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

np.random.seed(42); random.seed(42); torch.manual_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | device: {DEVICE}")

## 1. Concept: Why Language-Conditioned Long-Horizon Control Is Hard

A general-purpose robot in a home must do many tasks, specified the way humans naturally specify them — with **unconstrained language**, chained over a long horizon.

Prior multi-task methods specify the task via:
- **Goal images** — show the robot a picture of the desired end state. Impractical: a non-expert can't produce goal images on demand.
- **One-hot skill selectors** — pick task #7 from a fixed list. Doesn't scale and isn't how humans communicate.

Language is the natural interface, but it introduces three coupled challenges that CALVIN bundles together for the first time:

1. **Grounding** — relate words ('blue block', 'drawer') to multimodal perception.
2. **Long-horizon composition** — chain 5+ subtasks where each depends on the last.
3. **Continuous control** — output 7-DoF motor commands, not discrete primitives.

The most similar prior benchmark, ALFRED, uses 7 predefined discrete action primitives. CALVIN instead requires a learned repertoire of continuous skills.

In [ ]:
# == Cell 1: Goal-image / one-hot vs language specification ==
# Illustrate why language scales: one encoder generalises to unseen phrasings,
# while a one-hot selector has a fixed, closed task set.

tasks = ["open drawer", "push blue block left", "turn on led", "lift red block"]

# one-hot: each task is an isolated index, no relationship between them
onehot = np.eye(len(tasks))

# language: synonyms map to nearby points -> generalises to NEW phrasings
def toy_lang_embed(text, dim=16):
    rng = np.random.default_rng(abs(hash(text)) % (2**32))
    return rng.standard_normal(dim)

# a NEW phrasing of an existing task
queries = {"go open the drawer": "open drawer",
           "slide the blue block to the left": "push blue block left"}

print("One-hot selector: can only express the 4 fixed tasks. A new phrasing -> no slot.\n")
print("Language embeddings: a never-seen phrasing still lands near its task.")
for new_phrase, true_task in queries.items():
    e_new = toy_lang_embed(new_phrase)
    sims = {t: float(np.dot(e_new, toy_lang_embed(t)) /
                     (np.linalg.norm(e_new)*np.linalg.norm(toy_lang_embed(t)))) for t in tasks}
    # NOTE: random embeddings won't truly cluster; a real model (MiniLM) does.
    print(f"  '{new_phrase}'  (true: {true_task})")
print("\n(With a real sentence model like MiniLM, synonyms genuinely cluster — shown in Cell 3.)")

## 2. Concept: The Observation and Action Spaces

CALVIN gives a rich, flexible sensor suite — researchers choose which modalities to use.

**Observations:**

| Modality | Shape |
|---|---|
| RGB static camera | 200 x 200 x 3 |
| Depth static camera | 200 x 200 |
| RGB gripper camera | 84 x 84 x 3 |
| Depth gripper camera | 84 x 84 |
| Tactile image | 120 x 160 x 2 |
| Proprioception | EE pos (3) + orn (3) + gripper width (1) + joints (7) + gripper action (1) |

**Actions** (pick one space):
- Absolute cartesian EE pose (world frame): pos (3) + orn (3) + gripper (1)
- Relative cartesian displacement (gripper frame): pos (3) + orn (3) + gripper (1)
- Joint action: joint positions (7) + gripper (1)

**Design insight:** the static camera + absolute actions suit large traversals; the gripper camera + relative actions suit fine control like stacking. The paper encourages mixing modalities per task.

In [ ]:
# == Cell 2: Generate a synthetic CALVIN observation ==

def make_calvin_obs(seed=0):
    """Toy stand-in for CALVIN's multimodal observation (Fig. 2 in the paper)."""
    rng = np.random.default_rng(seed)
    return {
        "rgb_static":    rng.integers(0, 255, (200, 200, 3), dtype=np.uint8),
        "depth_static":  rng.uniform(0, 1, (200, 200)).astype(np.float32),
        "rgb_gripper":   rng.integers(0, 255, (84, 84, 3), dtype=np.uint8),
        "depth_gripper": rng.uniform(0, 1, (84, 84)).astype(np.float32),
        "tactile":       rng.uniform(0, 1, (120, 160, 2)).astype(np.float32),
        # proprio = ee_pos(3)+ee_orn(3)+gripper_width(1)+joints(7)+gripper_act(1) = 15
        "proprio":       rng.uniform(-1, 1, 15).astype(np.float32),
    }

obs = make_calvin_obs(seed=3)

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
axes[0].imshow(obs["rgb_static"]);             axes[0].set_title("RGB static (200x200)"); axes[0].axis("off")
axes[1].imshow(obs["depth_static"], cmap="viridis"); axes[1].set_title("Depth static"); axes[1].axis("off")
axes[2].imshow(obs["rgb_gripper"]);            axes[2].set_title("RGB gripper (84x84)"); axes[2].axis("off")
axes[3].imshow(obs["tactile"][:, :, 0], cmap="magma"); axes[3].set_title("Tactile (ch 0)"); axes[3].axis("off")
plt.tight_layout(); plt.show()

print("Observation modalities:", list(obs.keys()))
print("Proprioceptive vector (15-dim):", obs["proprio"].round(2))

# action spaces
action_spaces = {
    "abs_cartesian": 7,   # pos3 + orn3 + gripper1
    "rel_cartesian": 7,
    "joint":         8,   # joints7 + gripper1
}
print("\nAvailable action spaces (dims):", action_spaces)

## 3. Concept: Encoding Language Instructions

CALVIN ships precomputed **MiniLM** sentence embeddings. MiniLM distills a large Transformer language model, has a 30,522-word vocabulary, and maps any sentence to a **384-dim vector**. Crucially, synonymous instructions land near each other in this space — that's what lets a policy generalise to phrasings it never saw in training.

The paper also flags a hard sub-problem: the embeddings for "red block" and "blue block" are *too similar*, so the baseline confuses block colours. We'll see that in the improvements section.

We can't download MiniLM weights here, but `sentence-transformers` provides it if you want the real thing. The cell below tries the real model and falls back to a deterministic toy encoder so the notebook always runs.

In [ ]:
# == Cell 3: Language encoder (real MiniLM if available, else toy fallback) ==

USE_REAL_MINILM = False   # set True in Colab to: !pip install sentence-transformers

if USE_REAL_MINILM:
    from sentence_transformers import SentenceTransformer
    _model = SentenceTransformer("all-MiniLM-L6-v2")
    def encode_language(sentences):
        return _model.encode(sentences, normalize_embeddings=True)
else:
    # Deterministic toy encoder: shared word vectors so synonyms partially cluster
    _vocab = {}
    def _word_vec(w, dim=384):
        if w not in _vocab:
            rng = np.random.default_rng(abs(hash(w)) % (2**32))
            _vocab[w] = rng.standard_normal(dim).astype(np.float32)
        return _vocab[w]
    def encode_language(sentences):
        out = []
        for s in sentences:
            words = s.lower().replace(".", "").split()
            v = np.mean([_word_vec(w) for w in words], axis=0)
            out.append(v / (np.linalg.norm(v) + 1e-8))
        return np.stack(out)

# Demonstrate clustering of synonymous instructions
instructions = [
    "open the drawer", "go open the drawer", "pull the drawer handle",
    "turn on the led", "press the button to turn on the green light",
    "lift the red block", "pick up the red block",
]
emb = encode_language(instructions)
sim = emb @ emb.T

plt.figure(figsize=(7, 6))
plt.imshow(sim, cmap="magma", vmin=0, vmax=1)
plt.colorbar(label="Cosine similarity")
plt.xticks(range(len(instructions)), range(len(instructions)))
plt.yticks(range(len(instructions)), instructions, fontsize=9)
plt.title("Language instruction similarity (toy encoder)")
plt.tight_layout(); plt.show()

print("Instruction embedding dim:", emb.shape[1])
print("Drawer synonyms (0,1) similarity:", round(float(sim[0,1]), 3))
print("Drawer vs led (0,3) similarity:  ", round(float(sim[0,3]), 3))

## 4. Concept: Learning From Unstructured Play

CALVIN's training data is **24 hours of teleoperated 'play'** — three untrained people exploring four environments with a VR headset, told only to "explore without dropping objects". No predefined tasks.

**Why play data?**
- Task-agnostic, diverse, and cheap to collect.
- Covers the **multimodal** space of solutions (many ways to do a thing), including retrying and sub-optimal behaviour — which curated expert demos miss.

**The trick: goal relabeling.** Take any short window of the play stream. Treat its final state as a 'reached goal', and the preceding states/actions as optimal behaviour for reaching that goal. This turns unlabelled play into millions of goal-conditioned training pairs — no task labels needed. Only 1% of windows additionally get a language label.

In [ ]:
# == Cell 4: Goal relabeling on a synthetic play stream ==

def make_play_stream(T=400, state_dim=8, action_dim=7, seed=0):
    """A long unstructured stream of (state, action) from undirected play."""
    rng = np.random.default_rng(seed)
    states  = np.cumsum(0.05 * rng.standard_normal((T, state_dim)), axis=0).astype(np.float32)
    actions = np.diff(states, axis=0, prepend=states[:1])
    return states, actions[:, :action_dim] if action_dim <= state_dim else actions

def relabel_windows(states, actions, window=16, n_samples=5, rng=None):
    """Sample random windows; final state of each becomes the 'goal'.
    Returns (trajectory, goal_state) pairs for goal-conditioned imitation."""
    rng = rng or np.random.default_rng(0)
    T = len(states)
    pairs = []
    for _ in range(n_samples):
        start = rng.integers(0, T - window)
        traj_s = states[start:start + window]
        traj_a = actions[start:start + window]
        goal   = traj_s[-1]                         # <-- the relabeled goal
        pairs.append((traj_s, traj_a, goal))
    return pairs

states, actions = make_play_stream()
pairs = relabel_windows(states, actions, window=16, n_samples=4)

print(f"Play stream length: {len(states)} steps (no task labels)")
print(f"Relabeled into {len(pairs)} goal-conditioned windows\n")
for i, (s, a, g) in enumerate(pairs):
    print(f"  window {i}: traj {s.shape}, actions {a.shape}, goal-state dim {g.shape[0]}")

# visualise: the play trajectory + the relabeled goals
plt.figure(figsize=(8, 5))
plt.plot(states[:, 0], states[:, 1], color="lightgray", linewidth=1, label="Play stream")
for i, (s, a, g) in enumerate(pairs):
    plt.plot(s[:, 0], s[:, 1], linewidth=2)
    plt.scatter(g[0], g[1], s=80, zorder=5, label=f"relabeled goal {i}")
plt.xlabel("state dim 0"); plt.ylabel("state dim 1")
plt.title("Goal relabeling: any window's end-state becomes a training goal")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

## 5. Concept: The MCIL Baseline (Multi-Context Imitation Learning)

The paper's baseline is **MCIL** (Lynch & Sermanet, 2021). The key ideas:

1. Train a single goal-conditioned policy $\pi_\theta(a_t \mid x_t, z)$ over relabeled play data.
2. Handle the multimodality of free-form data with a sequence-to-sequence **conditional VAE** that auto-encodes demonstrations through a latent 'plan' space $z$.
3. Support multiple *contexts* for the goal — a goal image OR a language instruction — sharing one policy, with one encoder per context type.

Because goal images and language can be trained jointly, control is learned mostly from cheap unlabelled play, and language annotation is needed for under 1% of the data.

We'll build a simplified language-conditioned policy (dropping the CVAE plan latent for clarity) to see the mechanics.

In [ ]:
# == Cell 5: A simplified language-conditioned policy ==

class LangConditionedPolicy(nn.Module):
    """Simplified MCIL-style policy: pi(a | obs, language).
    Real MCIL adds a seq2seq CVAE 'plan' latent; we drop it for clarity."""
    def __init__(self, obs_dim, lang_dim, action_dim, hidden=128):
        super().__init__()
        self.obs_enc  = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU())
        self.lang_enc = nn.Sequential(nn.Linear(lang_dim, hidden), nn.ReLU())
        self.head     = nn.Sequential(
            nn.Linear(2 * hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, action_dim))

    def forward(self, obs, lang):
        h = torch.cat([self.obs_enc(obs), self.lang_enc(lang)], dim=-1)
        return self.head(h)


OBS_DIM, LANG_DIM, ACTION_DIM = 15, 384, 7
policy = LangConditionedPolicy(OBS_DIM, LANG_DIM, ACTION_DIM).to(DEVICE)

# synthetic supervised demo: (obs, language) -> action
def make_demo_batch(n=512):
    rng = np.random.default_rng(0)
    obs  = rng.standard_normal((n, OBS_DIM)).astype(np.float32)
    instr = rng.integers(0, len(instructions), n)
    lang = emb[instr].astype(np.float32)
    # ground-truth action is a fixed (unknown to policy) function of obs+task
    W = rng.standard_normal((OBS_DIM + LANG_DIM, ACTION_DIM)).astype(np.float32) * 0.1
    act = np.concatenate([obs, lang], axis=1) @ W
    return (torch.tensor(obs).to(DEVICE), torch.tensor(lang).to(DEVICE),
            torch.tensor(act.astype(np.float32)).to(DEVICE))

obs_b, lang_b, act_b = make_demo_batch()
opt = optim.Adam(policy.parameters(), lr=1e-3)
losses = []
for step in range(300):
    pred = policy(obs_b, lang_b)
    loss = nn.functional.mse_loss(pred, act_b)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())

plt.figure(figsize=(8, 3))
plt.plot(losses); plt.xlabel("Step"); plt.ylabel("Imitation MSE")
plt.title("Language-conditioned behaviour cloning")
plt.tight_layout(); plt.show()
print(f"Final imitation loss: {losses[-1]:.4f}")

## 6. Concept: The Challenge Tiers and Long-Horizon Metric

CALVIN evaluates with three environment combinations of increasing difficulty:

| Tier | Train -> Test | Why it's hard |
|---|---|---|
| Single | D -> D | baseline; same room |
| Multi | A,B,C,D -> D | generalise across textures/layouts |
| Zero-shot | A,B,C -> D | test room never seen in training |

**Long-Horizon MTLC metric.** The 34 tasks become subgoals. The agent is given **5 language instructions in a row** (1000 unique chains). It advances to the next instruction only when the current subtask succeeds, detected by comparing environment state between first and last frame. Reported numbers: fraction completing >= 1, 2, 3, 4, 5 tasks in sequence.

This compounding is brutal: if each subtask succeeds at rate p, completing all 5 is roughly p^5.

In [ ]:
# == Cell 6: The long-horizon chaining metric ==

def evaluate_chain(subtask_success_probs, rng):
    """Walk a 5-instruction chain; stop at the first failure.
    Returns how many consecutive subtasks were completed."""
    completed = 0
    for p in subtask_success_probs:
        if rng.random() < p:
            completed += 1
        else:
            break
    return completed

def run_lh_mtlc(per_task_success, n_chains=1000, chain_len=5, seed=0):
    rng = np.random.default_rng(seed)
    probs = [per_task_success] * chain_len
    chains = [evaluate_chain(probs, rng) for _ in range(n_chains)]
    return {k: round(float(np.mean([c >= k for c in chains])), 4) for k in range(1, chain_len + 1)}

# Mimic the paper: single-task success ~0.54 in the easiest tier
tiers = {
    "Single (D->D)":     0.539,
    "Multi (ABCD->D)":   0.356,
    "Zero-shot (ABC->D)":0.386,
}

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(1, 6)
for name, p in tiers.items():
    res = run_lh_mtlc(p)
    ax.plot(x, [res[k] for k in x], "o-", label=name)
    print(f"{name}: " + "  ".join(f"{k}->{run_lh_mtlc(p)[k]:.3f}" for k in x))
ax.set_xticks(x)
ax.set_xlabel("Number of instructions completed in a row")
ax.set_ylabel("Fraction of chains")
ax.set_title("Long-horizon chaining: success collapses with sequence length")
ax.set_yscale("log"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("\nEven at 54% single-task success, completing 5 in a row is near-zero —")
print("matching the paper's headline finding (0.08% for the best MCIL model).")

---
# Part II: Research Improvements

CALVIN's baseline (MCIL) is weak by design — the paper explicitly invites better methods. The next three sections propose improvements with working code, each motivated by a specific failure the paper reports.

| # | Failure in the paper | Proposed improvement |
|---|---|---|
| 1 | Baseline confuses red vs blue blocks (language embeddings too similar) | **Vision-language contrastive grounding** (CLIP-style) to align words with the right object |
| 2 | Long-horizon success collapses (0.08% on 5-chains) | **Hierarchical sub-goal planner** with per-subtask replanning |
| 3 | Sparse reward (+1 on completion) makes long-horizon RL hard | **Progress-based dense reward** from the 34-task state detector |

Each connects to active research the paper itself points toward (CLIP alignment, hierarchical policies, reward shaping).

## 7. Improvement 1: Vision-Language Contrastive Grounding

**The gap.** The paper observes: "the policy sometimes correctly executes block manipulation tasks, but confuses the red and blue block colors" because MiniLM embeds 'red' and 'blue' sentences too similarly. The language pathway alone can't resolve which block is meant.

**The improvement.** Add a **CLIP-style contrastive objective** that aligns the visual embedding of the *correct* object with its language instruction, and pushes mismatched pairs apart. This forces the model to ground 'red block' to the actual red pixels, not just the word. The paper explicitly suggests this: backpropagating through the language model and adding visual-language alignment losses.

Below we train a tiny image-text aligner and show that a contrastive loss separates 'red block' from 'blue block' where the raw language embeddings don't.

In [ ]:
# == Cell 7: CLIP-style vision-language alignment ==

class VisionLangAligner(nn.Module):
    """Project image features and language into a shared space where
    matching (image, instruction) pairs have high cosine similarity."""
    def __init__(self, img_dim, lang_dim, emb=64):
        super().__init__()
        self.img_proj  = nn.Linear(img_dim, emb)
        self.lang_proj = nn.Linear(lang_dim, emb)
    def forward(self, img_feat, lang_feat):
        zi = nn.functional.normalize(self.img_proj(img_feat), dim=-1)
        zl = nn.functional.normalize(self.lang_proj(lang_feat), dim=-1)
        return zi, zl

def clip_loss(zi, zl, temp=0.07):
    """Symmetric InfoNCE: diagonal pairs match, off-diagonal don't."""
    logits = zi @ zl.T / temp
    labels = torch.arange(len(zi), device=zi.device)
    return 0.5 * (nn.functional.cross_entropy(logits, labels) +
                  nn.functional.cross_entropy(logits.T, labels))

# Synthetic 'red block' vs 'blue block' scenario.
# Image features DO differ by colour; language embeddings barely do.
rng = np.random.default_rng(0)
IMG_DIM = 32
red_img  = rng.standard_normal((64, IMG_DIM)).astype(np.float32) + np.array([2.0] + [0]*(IMG_DIM-1), np.float32)
blue_img = rng.standard_normal((64, IMG_DIM)).astype(np.float32) + np.array([-2.0] + [0]*(IMG_DIM-1), np.float32)
red_txt  = np.tile(encode_language(["lift the red block"]),  (64, 1)).astype(np.float32)
blue_txt = np.tile(encode_language(["lift the blue block"]), (64, 1)).astype(np.float32)

img  = torch.tensor(np.concatenate([red_img,  blue_img]))
txt  = torch.tensor(np.concatenate([red_txt,  blue_txt]))

aligner = VisionLangAligner(IMG_DIM, LANG_DIM).to(DEVICE)
opt = optim.Adam(aligner.parameters(), lr=1e-3)
for step in range(400):
    idx = torch.randperm(len(img))[:32]
    zi, zl = aligner(img[idx].to(DEVICE), txt[idx].to(DEVICE))
    loss = clip_loss(zi, zl)
    opt.zero_grad(); loss.backward(); opt.step()

# After training: does 'red block' text match red images more than blue?
aligner.eval()
with torch.no_grad():
    zi_red,  zl_red  = aligner(torch.tensor(red_img).to(DEVICE),  torch.tensor(red_txt).to(DEVICE))
    zi_blue, _       = aligner(torch.tensor(blue_img).to(DEVICE), torch.tensor(blue_txt).to(DEVICE))
    red_text_to_red  = (zl_red[0] @ zi_red.T).mean().item()
    red_text_to_blue = (zl_red[0] @ zi_blue.T).mean().item()

raw_lang_sim = float(encode_language(["lift the red block"])[0] @ encode_language(["lift the blue block"])[0])
print(f"Raw language similarity (red vs blue instruction): {raw_lang_sim:.3f}  <- too high, causes confusion")
print(f"After grounding: 'red block' text -> red images:  {red_text_to_red:.3f}")
print(f"After grounding: 'red block' text -> blue images: {red_text_to_blue:.3f}")
print(f"\nGrounding gap: {red_text_to_red - red_text_to_blue:.3f} (larger = colours disambiguated)")

## 8. Improvement 2: Hierarchical Sub-Goal Planner with Replanning

**The gap.** The best MCIL model completes 5-instruction chains only 0.08% of the time. A flat policy commits to the whole chain with no recovery — one subtask failure ends the episode.

**The improvement.** Treat each language instruction as an explicit **subgoal** and give the agent a chance to **replan / retry** when a subtask fails, instead of aborting the whole chain. This is the core idea behind hierarchical and recovery-capable policies. CALVIN's per-subtask success detector makes this natural: after each subtask, check completion; on failure, attempt one recovery before moving on.

We compare a flat chain (abort on first failure) against a replanning chain (one retry per subtask).

In [ ]:
# == Cell 8: Flat vs hierarchical (replanning) long-horizon execution ==

def chain_with_optional_replan(subtask_probs, rng, n_retries=0):
    """Walk a chain. On a subtask failure, allow up to n_retries recovery attempts
    before aborting. Returns number of subtasks completed."""
    completed = 0
    for p in subtask_probs:
        success = False
        for _attempt in range(1 + n_retries):
            if rng.random() < p:
                success = True
                break
        if success:
            completed += 1
        else:
            break
    return completed

per_task = 0.6
chain_len = 5
probs = [per_task] * chain_len
N = 5000

def measure(n_retries):
    rng = np.random.default_rng(1)
    chains = [chain_with_optional_replan(probs, rng, n_retries) for _ in range(N)]
    return [float(np.mean([c >= k for c in chains])) for k in range(1, chain_len + 1)]

flat   = measure(n_retries=0)
replan = measure(n_retries=1)
replan2= measure(n_retries=2)

x = np.arange(1, chain_len + 1)
plt.figure(figsize=(9, 5))
plt.plot(x, flat,    "o-", color="tomato",    label="Flat policy (no recovery)")
plt.plot(x, replan,  "s-", color="steelblue", label="Hierarchical (1 retry / subtask)")
plt.plot(x, replan2, "^-", color="teal",      label="Hierarchical (2 retries / subtask)")
plt.xticks(x); plt.xlabel("Instructions completed in a row")
plt.ylabel("Fraction of chains"); plt.title("Replanning dramatically improves long-horizon success")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print(f"Complete all 5 in a row:")
print(f"  flat:            {flat[-1]:.3f}")
print(f"  +1 retry/subtask:{replan[-1]:.3f}")
print(f"  +2 retry/subtask:{replan2[-1]:.3f}")
print("\nThe same per-subtask skill, wrapped in recovery, multiplies end-to-end success.")

## 9. Improvement 3: Progress-Based Dense Reward

**The gap.** CALVIN's reward is sparse: tasks are detected as complete by a state change between first and last frame, giving a single +1. For RL agents on long-horizon chains, this signal is almost never encountered.

**The improvement.** CALVIN already provides a **per-task state detector for all 34 tasks**. We can expose a dense **progress reward** = fraction of subgoals in the current chain that are satisfied. This gives a learning signal at every subtask boundary without changing the official success metric (which stays binary for reporting).

We verify with a small tabular Q-learning experiment that dense progress reward solves a 5-subgoal chain far faster than sparse reward.

In [ ]:
# == Cell 9: Sparse vs progress reward (tabular Q-learning on a subgoal chain) ==

def sparse_reward(progress, total):    return 1.0 if progress == total else 0.0
def progress_reward(progress, total):  return progress / total

def run_q_learning(reward_fn, n_states=6, episodes=500, alpha=0.5, gamma=0.95, eps=0.2, seed=0):
    """States 0..5 represent subgoals completed. Action 1 advances, 0 stalls.
    Reward is the delta of the chosen reward function between states."""
    rng = np.random.default_rng(seed)
    Q = np.zeros((n_states, 2))
    goal = n_states - 1
    curve = []
    for ep in range(episodes):
        s = 0
        for _ in range(25):
            a = rng.integers(2) if rng.random() < eps else int(np.argmax(Q[s]))
            s_next = min(s + 1, goal) if a == 1 else s
            r = reward_fn(s_next, goal) - reward_fn(s, goal)
            Q[s, a] += alpha * (r + gamma * Q[s_next].max() - Q[s, a])
            s = s_next
            if s == goal:
                break
        # greedy rollout for evaluation
        s = 0
        for _ in range(25):
            s = min(s + 1, goal) if int(np.argmax(Q[s])) == 1 else s
            if s == goal:
                break
        curve.append(1.0 if s == goal else 0.0)
    w = 25
    return np.convolve(curve, np.ones(w) / w, mode="valid")

sparse_curve = run_q_learning(sparse_reward)
dense_curve  = run_q_learning(progress_reward)

plt.figure(figsize=(9, 4))
plt.plot(sparse_curve, color="tomato",    label="Sparse reward (+1 at chain end)")
plt.plot(dense_curve,  color="steelblue", label="Progress reward (per subgoal)")
plt.xlabel("Episode"); plt.ylabel("Success rate (smoothed)")
plt.title("Progress reward learns the 5-subgoal chain much faster")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print("Sparse reward: agent rarely stumbles onto the single terminal reward.")
print("Progress reward: each subgoal gives partial credit -> fast, stable learning.")
print("\nKeep the binary metric for REPORTING; use dense reward only for TRAINING.")

## 10. Summary

**Part I — what CALVIN is:**
- The first benchmark combining unconstrained language, multimodal sensing, 7-DoF continuous control, and long-horizon task chaining.
- Four related environments (A-D) with a Franka Panda, drawer, slider, button, switch, and coloured blocks.
- 24h of unstructured play data, only 1% language-labelled, with precomputed MiniLM embeddings.
- Goal relabeling turns unlabelled play into goal-conditioned training pairs.
- Three difficulty tiers (single / multi / zero-shot) and a brutal long-horizon chaining metric.
- The MCIL baseline scores 53.9% on single tasks but ~0.08% on 5-task chains — lots of headroom.

**Part II — improvements you can pitch:**
1. **Vision-language contrastive grounding** (CLIP-style) to fix the red/blue block confusion the paper reports.
2. **Hierarchical sub-goal planner with replanning** to stop one subtask failure from killing the whole chain.
3. **Progress-based dense reward** from the existing 34-task detector, making long-horizon RL tractable while keeping the binary success metric for reporting.

**How to frame it to your advisor:** CALVIN exposes three bottlenecks — perceptual grounding, long-horizon credit assignment, and recovery from failure. These three improvements target one bottleneck each, and all three are buildable on top of the released benchmark without new data collection.

## References

- Mees, Hermann, Rosete-Beas & Burgard (2022). *CALVIN: A Benchmark for Language-Conditioned Policy Learning for Long-Horizon Robot Manipulation Tasks.* IEEE RA-L. arXiv:2112.03227.
- Lynch & Sermanet (2021). *Language Conditioned Imitation Learning over Unstructured Data.* RSS.
- Lynch et al. (2019). *Learning Latent Plans from Play.* CoRL.
- Radford et al. (2021). *Learning Transferable Visual Models from Natural Language Supervision (CLIP).* ICML.
- Wang et al. (2020). *MiniLM: Deep Self-Attention Distillation.* NeurIPS.